Switched from CIFAR-10 to AG_News (4-class news classification)

GRU, RNN, BiRNN, LSTM

In [1]:
#%pip install -q datasets
from datasets import load_dataset

ds = load_dataset("ag_news")
ds["train"][0]


c:\Users\Nutzer\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py:90: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.2)
  return _bootstrap._gcd_import(name[level:], package, level)


{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [2]:
# Install if needed (skip if already installed)
# %pip install -q datasets torch

import re
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset


# ----------------------------
# Data loading (Hugging Face datasets)
# ----------------------------
ds = load_dataset("ag_news")  # ds["train"], ds["test"]

class AGNewsHFDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # HF ag_news labels are 0..3 (unlike torchtext which uses 1..4)
        return int(item["label"]), item["text"]


train_ds = AGNewsHFDataset(ds["train"])
test_ds  = AGNewsHFDataset(ds["test"])


# ----------------------------
# Tokenizer (simple basic_english equivalent without torchtext)
# ----------------------------
def basic_english_tokenizer(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # replace symbols with spaces
    text = re.sub(r"\s+", " ", text).strip()    # collapse whitespace
    return text.split()


# ----------------------------
# Vocabulary (built without torchtext)
# ----------------------------
class Vocab:
    def __init__(self, stoi, itos, default_index: int):
        self.stoi = stoi
        self.itos = itos
        self.default_index = default_index

    def __len__(self):
        return len(self.itos)

    def __getitem__(self, token: str):
        return self.stoi.get(token, self.default_index)

    def __call__(self, tokens):
        return [self[token] for token in tokens]


def build_vocab_from_texts(text_iter, tokenizer, specials=("<unk>", "<pad>"), min_freq=2):
    counter = Counter()
    for text in text_iter:
        toks = tokenizer(text)
        if not toks:
            toks = ["<unk>"]
        counter.update(toks)

    itos = list(specials)
    # sort by freq desc, then token asc for stability
    for tok, freq in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
        if freq >= min_freq and tok not in specials:
            itos.append(tok)

    stoi = {tok: i for i, tok in enumerate(itos)}
    default_index = stoi["<unk>"]
    return Vocab(stoi=stoi, itos=itos, default_index=default_index)


# Build vocab from training set
vocab = build_vocab_from_texts(
    (ds["train"][i]["text"] for i in range(len(ds["train"]))),
    tokenizer=basic_english_tokenizer,
    specials=("<unk>", "<pad>"),
    min_freq=2
)
pad_idx = vocab["<pad>"]


def text_pipeline(x: str):
    tokens = basic_english_tokenizer(x)
    if not tokens:
        tokens = ["<unk>"]
    return vocab(tokens)

def label_pipeline(y: int):
    # HF ag_news labels are already 0..3
    return int(y)


# ----------------------------
# Collate function for DataLoader (padding + lengths)
# ----------------------------
def collate_batch(batch):
    labels = []
    sequences = []
    lengths = []

    for (label, text) in batch:
        labels.append(label_pipeline(label))
        ids = torch.tensor(text_pipeline(text), dtype=torch.long)
        sequences.append(ids)
        lengths.append(ids.size(0))

    labels = torch.tensor(labels, dtype=torch.long)
    lengths = torch.tensor(lengths, dtype=torch.long)
    tokens_padded = pad_sequence(sequences, batch_first=True, padding_value=pad_idx)
    return tokens_padded, lengths, labels


batch_size = 64
trainloader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_batch, num_workers=0)
testloader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_batch, num_workers=0)


## Models

In [3]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

# --------
# GRU
# --------
class NetGRU(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        last_hidden = h_n[-1]                 # (N, H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits


# --------
# RNN (unidirectional)
# --------
class NetRNN(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embed_dim, hidden_size, num_layers=1, batch_first=True, nonlinearity="tanh")
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        last_hidden = h_n[-1]                 # (N, H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits


# --------
# BiRNN (bidirectional RNN)
# --------
class NetBiRNN(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(
            embed_dim, hidden_size, num_layers=1, batch_first=True,
            bidirectional=True, nonlinearity="tanh"
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # 2H input because bidirectional

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        # h_n: (layers*dirs, N, H) = (2, N, H); last two are forward and backward
        forward_last = h_n[-2]                # (N, H)
        backward_last = h_n[-1]               # (N, H)
        last_hidden = torch.cat([forward_last, backward_last], dim=1)  # (N, 2H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits


# --------
# LSTM (unidirectional)
# --------
class NetLSTM(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, c_n) = self.lstm(packed)
        last_hidden = h_n[-1]                 # (N, H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits


In [ ]:
import torch.optim as optim


def main(NetClass, epochs=3, lr=0.05, momentum=0.9):
    """
    NetClass: one of NetGRU / NetRNN / NetBiRNN / NetLSTM
    Uses global trainloader, testloader, vocab, pad_idx defined in cell 3.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    net       = NetClass(vocab_size=len(vocab), pad_idx=pad_idx).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum)

    # Training loop
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0

        for i, (tokens, lengths, labels) in enumerate(trainloader, 1):
            tokens  = tokens.to(device)
            lengths = lengths.to(device)
            labels  = labels.to(device)

            optimizer.zero_grad()
            outputs = net(tokens, lengths)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 200 == 0:
                print(f"[{NetClass.__name__}  epoch {epoch+1}, step {i:5d}]  loss: {running_loss/200:.3f}")
                running_loss = 0.0

    print(f"{NetClass.__name__}: Finished Training")

    # Evaluation
    net.eval()
    correct = total = 0

    with torch.no_grad():
        for tokens, lengths, labels in testloader:
            tokens  = tokens.to(device)
            lengths = lengths.to(device)
            labels  = labels.to(device)

            outputs = net(tokens, lengths)
            preds   = outputs.argmax(dim=1)
            total   += labels.size(0)
            correct += (preds == labels).sum().item()

    print(f"{NetClass.__name__}: Test Accuracy: {100.0 * correct / total:.2f}%")
    return net


In [ ]:
# GRU
main(NetGRU)

In [ ]:

# RNN
main(NetRNN)

In [ ]:
# BiRNN
main(NetBiRNN)

In [ ]:





# LSTM
main(NetLSTM)


[1,   200] loss: 1.337
[1,   400] loss: 1.118
[1,   600] loss: 0.800
[1,   800] loss: 0.625
[1,  1000] loss: 0.536
[1,  1200] loss: 0.490
[1,  1400] loss: 0.463
[1,  1600] loss: 0.441
[1,  1800] loss: 0.435
[2,   200] loss: 0.379
[2,   400] loss: 0.381
[2,   600] loss: 0.376
[2,   800] loss: 0.374
[2,  1000] loss: 0.362
[2,  1200] loss: 0.365
[2,  1400] loss: 0.360
[2,  1600] loss: 0.353
[2,  1800] loss: 0.351
Finished Training
Test Accuracy: 87.51%
[1,   200] loss: 1.369
[1,   400] loss: 1.345
[1,   600] loss: 1.346
[1,   800] loss: 1.350
[1,  1000] loss: 1.398
[1,  1200] loss: 1.476
[1,  1400] loss: 1.590
[1,  1600] loss: 1.656
[1,  1800] loss: 1.615
[2,   200] loss: 1.589
[2,   400] loss: 1.673
[2,   600] loss: 1.658
[2,   800] loss: 1.728
[2,  1000] loss: 1.653
[2,  1200] loss: 1.691
[2,  1400] loss: 1.682
[2,  1600] loss: 1.573
[2,  1800] loss: 1.592
Finished Training
Test Accuracy: 25.92%
[1,   200] loss: 1.354
[1,   400] loss: 1.319
[1,   600] loss: 1.284
[1,   800] loss: 1.284


NetLSTM(
  (embedding): Embedding(44118, 64, padding_idx=1)
  (lstm): LSTM(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
)

## BiRNN code from the textbook (white book)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


# =========================================================
# p.191-192: Bidirectional RNN (BiRNN)  -- fill-in-the-blank answers applied
#   (a) = hidden_size * 2
#   (c) = unsqueeze(-1)
# =========================================================
class BiRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.RNN = nn.RNN(
            input_size, hidden_size, num_layers,
            batch_first=True, bidirectional=True
        )  # bidirectional RNN

        # (a) = hidden_size * 2
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # fully connected layer

    def forward(self, x, seq_lengths, masks):
        """
        x:           (N, T, input_size)  -- expects pre-embedded input
        seq_lengths: (N,)
        masks:       (N, T)  1=valid, 0=pad
        """

        # sort by sequence length (required for pack_padded_sequence)
        seq_lengths, perm_idx = seq_lengths.sort(0, descending=True)
        x = x[perm_idx]
        masks = masks[perm_idx]

        # pack padded sequences
        x = pack_padded_sequence(x, seq_lengths.cpu(), batch_first=True, enforce_sorted=True)

        # initial hidden state (x2 for bidirectional)
        h0 = x.data.new_zeros(self.num_layers * 2, masks.size(0), self.hidden_size)

        # pass through RNN
        out, _ = self.RNN(x, h0)

        # unpack
        out, _ = pad_packed_sequence(out, batch_first=True)  # (N, T, 2H)

        # restore original order
        _, unperm_idx = perm_idx.sort(0)
        out = out[unperm_idx]
        masks = masks[unperm_idx]

        # mask out padding positions  (c) = unsqueeze(-1)
        out = out * masks.unsqueeze(-1)  # (N, T, 2H)

        # --- alternative: sum forward + backward instead of concat ---
        # out = out[:, :, :self.hidden_size] + out[:, :, self.hidden_size:]  # (N, T, H)
        # self.fc = nn.Linear(self.hidden_size, num_classes)

        out = self.fc(out)  # (N, T, num_classes)
        return out


# =========================================================
# p.192: BiRNNLoss  -- fill-in-the-blank answers applied
#   (d) = loss * masks
#   (e) = masks
# =========================================================
class BiRNNLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # reduction="none" is needed to apply masks per token
        self.loss_fn = nn.CrossEntropyLoss(reduction="none")

    def forward(self, outputs, targets, masks):
        """
        outputs: (N, T, C)
        targets: (N, T)
        masks:   (N, T)
        """
        targets = targets.view(-1)
        outputs = outputs.view(-1, outputs.size(-1))
        masks = masks.view(-1).float()

        loss = self.loss_fn(outputs, targets)               # (N*T,)
        loss = torch.sum(loss * masks) / torch.sum(masks)   # (d) / (e)
        return loss


# =========================================================
# Wrapper to connect BiRNN with the (tokens, lengths, labels) DataLoader format
#   - masks are generated automatically from tokens
# =========================================================
class NetBiRNNClassifier(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4, num_layers=1):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.birnn = BiRNN(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            num_classes=num_classes
        )

    def forward(self, tokens, lengths):
        # tokens: (N, T)
        x = self.embedding(tokens)              # (N, T, embed_dim)
        masks = (tokens != self.pad_idx).float()  # (N, T)

        # BiRNN returns (N, T, C)
        out = self.birnn(x, lengths, masks)

        # For sentence classification: take the last valid step -> (N, C)
        idx = (lengths - 1).clamp(min=0)        # (N,)
        last_logits = out[torch.arange(out.size(0), device=out.device), idx]
        return last_logits


# =========================================================
# Training + evaluation
# =========================================================
def main(NetClass):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    net = NetClass(vocab_size=len(vocab), pad_idx=pad_idx).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=0.05, momentum=0.9)

    for epoch in range(2):
        net.train()
        running_loss = 0.0

        for i, (tokens, lengths, labels) in enumerate(trainloader, 0):
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(tokens, lengths)  # (N, C)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 200 == 199:
                print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}")
                running_loss = 0.0

    print("Finished Training")

    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for tokens, lengths, labels in testloader:
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)
            outputs = net(tokens, lengths)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    print(f"Test Accuracy: {100.0 * correct / total:.2f}%")


# Usage: run with BiRNN wrapper
# main(NetBiRNNClassifier)


In [ ]:
main(NetBiRNNClassifier)

[1,   200] loss: 1.393
[1,   400] loss: 1.373
[1,   600] loss: 1.376
[1,   800] loss: 1.362
[1,  1000] loss: 1.329
[1,  1200] loss: 1.285
[1,  1400] loss: 1.259
[1,  1600] loss: 1.255
[1,  1800] loss: 1.195
[2,   200] loss: 1.165
[2,   400] loss: 1.141
[2,   600] loss: 1.165
[2,   800] loss: 1.173
[2,  1000] loss: 1.269
[2,  1200] loss: 1.208
[2,  1400] loss: 1.314
[2,  1600] loss: 1.364
[2,  1800] loss: 1.416
Finished Training
Test Accuracy: 30.80%
